In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf

In [2]:
data = [x for x in range(1, 21)]
print(data)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [3]:
tf_df = tf.data.Dataset.from_tensor_slices(data)

In [4]:
width = len(str(max(data)))

### Iterator

In [5]:
for itr in tf_df.as_numpy_iterator():
    print(f"{str(itr).zfill(width)}", end=" ")

01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 

### Head

In [6]:
for itr in tf_df.take(10).as_numpy_iterator():
    print(f"{str(itr).zfill(width)}", end=" ")


01 02 03 04 05 06 07 08 09 10 

### Filter

In [7]:
for itr in tf_df.filter(lambda x: x % 2 == 0).as_numpy_iterator():
    print(f"{str(itr).zfill(width)}", end=" ")

02 04 06 08 10 12 14 16 18 20 

### Map

In [8]:
for itr in tf_df.map(lambda x: x**2).as_numpy_iterator():
    print(f"{str(itr).zfill(width)}", end=" ")

01 04 09 16 25 36 49 64 81 100 121 144 169 196 225 256 289 324 361 400 

### Shuffle

In [9]:
for itr in tf_df.shuffle(5).as_numpy_iterator():
    print(f"{str(itr).zfill(width)}", end=" ")

01 04 06 08 02 10 09 03 05 07 14 11 15 13 16 18 12 19 20 17 

### Batch

In [10]:
for itr in tf_df.batch(3).as_numpy_iterator():
    print(itr)

[1 2 3]
[4 5 6]
[7 8 9]
[10 11 12]
[13 14 15]
[16 17 18]
[19 20]


#### Pipeline (filter -> map -> shuffle -> batch)

In [11]:
tf_df = (
    tf.data.Dataset.from_tensor_slices(data)
    .filter(lambda x: x % 2 == 0)
    .map(lambda y: y**2)
    .shuffle(2)
    .batch(3)
)

for itr in tf_df.as_numpy_iterator():
    print(itr)

[16  4 36]
[100 144  64]
[256 324 400]
[196]


# Flower classification pipeline

In [12]:
df = tf.data.Dataset.list_files(
    "/Users/sarveshmhadgut/.keras/datasets/flowers_dataset/flower_photos/*/*",
    shuffle=False,
)
type(df), len(df)

(tensorflow.python.data.ops.from_tensor_slices_op._TensorSliceDataset, 3670)

In [13]:
df = df.shuffle(4000)
for img in df.take(5):
    print(img)

tf.Tensor(b'/Users/sarveshmhadgut/.keras/datasets/flowers_dataset/flower_photos/tulips/8659691170_09db83d023.jpg', shape=(), dtype=string)
tf.Tensor(b'/Users/sarveshmhadgut/.keras/datasets/flowers_dataset/flower_photos/sunflowers/4942258704_c4146b710a_n.jpg', shape=(), dtype=string)
tf.Tensor(b'/Users/sarveshmhadgut/.keras/datasets/flowers_dataset/flower_photos/sunflowers/10386540696_0a95ee53a8_n.jpg', shape=(), dtype=string)
tf.Tensor(b'/Users/sarveshmhadgut/.keras/datasets/flowers_dataset/flower_photos/daisy/14021430525_e06baf93a9.jpg', shape=(), dtype=string)
tf.Tensor(b'/Users/sarveshmhadgut/.keras/datasets/flowers_dataset/flower_photos/tulips/4395433872_e073d8c721_n.jpg', shape=(), dtype=string)


In [14]:
train_size = int(len(df) * 0.8)
train_size

2936

In [15]:
train_df = df.take(train_size)
test_df = df.skip(train_size)
len(train_df), len(test_df)

(2936, 734)

In [16]:
class_names = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

label_lookup = tf.keras.layers.StringLookup(
    vocabulary=class_names,
    mask_token=None,
    num_oov_indices=0,
    output_mode="int",
)


In [17]:
def get_label(filepath):
    label = tf.strings.split(filepath, os.path.sep)
    return label[-2]

In [18]:
get_label(
    "/Users/sarveshmhadgut/.keras/datasets/flowers_dataset/flower_photos/tulips/4574785121_5d8ec4626e.jpg"
)

<tf.Tensor: shape=(), dtype=string, numpy=b'tulips'>

In [19]:
import cv2


def get_image(filepath):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (128, 128))
    img = tf.cast(img, tf.float32) / 255

    return img

In [20]:
print(
    get_image(
        "/Users/sarveshmhadgut/.keras/datasets/flowers_dataset/flower_photos/tulips/4574785121_5d8ec4626e.jpg"
    )
)

tf.Tensor(
[[[3.69357705e-01 4.83009964e-01 1.23646937e-01]
  [4.73110318e-01 5.40803552e-01 1.97140440e-01]
  [4.61175650e-01 5.46898723e-01 1.68439314e-01]
  ...
  [6.93615973e-01 0.00000000e+00 0.00000000e+00]
  [8.14107239e-01 1.62353518e-03 1.20705999e-02]
  [9.76343036e-01 3.27381343e-01 3.27146530e-01]]

 [[4.36141908e-01 5.27543128e-01 1.18571445e-01]
  [4.80147064e-01 5.37392795e-01 1.69194236e-01]
  [5.19674599e-01 5.77078998e-01 2.31403187e-01]
  ...
  [7.20104754e-01 0.00000000e+00 2.87990202e-03]
  [9.69977856e-01 1.38909787e-01 1.76332965e-01]
  [9.82240915e-01 2.92106122e-01 2.90707111e-01]]

 [[4.72358257e-01 5.51907897e-01 9.87146720e-02]
  [4.90216911e-01 5.51568031e-01 1.34707704e-01]
  [5.36239803e-01 5.96082509e-01 2.29862228e-01]
  ...
  [6.89959109e-01 8.13802126e-06 0.00000000e+00]
  [9.89973724e-01 2.54543155e-01 3.05390716e-01]
  [9.74005699e-01 2.54801661e-01 2.49441594e-01]]

 ...

 [[1.34360641e-01 2.44536281e-01 2.95201913e-02]
  [7.77769834e-02 1.67181998

In [21]:
def process_image(filepath):
    label = get_label(filepath)
    label = label_lookup(label)
    img = get_image(filepath)
    return img, label

In [22]:
import numpy as np

train_df = train_df.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)
test_df = test_df.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
train_df = train_df.batch(32).prefetch(tf.data.AUTOTUNE)
test_df = test_df.batch(32).prefetch(tf.data.AUTOTUNE)

In [24]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(128, 128, 3)),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(5, activation="softmax"),
    ]
)

In [25]:
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,285 (12.61 MB)

 Trainable params: 3,305,285 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
model.fit(train_df, epochs=10, validation_data=test_df)

Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.4162 - loss: 1.3567 - val_accuracy: 0.5300 - val_loss: 1.0678
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.5552 - loss: 1.0945 - val_accuracy: 0.6063 - val_loss: 0.9998
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.5933 - loss: 1.0421 - val_accuracy: 0.6022 - val_loss: 0.9571
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.6209 - loss: 1.0355 - val_accuracy: 0.6907 - val_loss: 0.8906
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.6202 - loss: 1.0358 - val_accuracy: 0.7411 - val_loss: 0.7220
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.6373 - loss: 1.1497 - val_accuracy: 0.7371 - val_loss: 0.7248
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.6621 - loss: 1.2164 - val_accuracy: 0.7711 - val_loss: 1.0377
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.6873 - loss: 1.2002 - val_accuracy: 0.7480 - v

In [27]:
model.evaluate(test_df)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7643 - loss: 1.1548


[1.1547969579696655, 0.7643051743507385]